# 2. NaiveRAG, the vector baseline

Vector retrieval-augmented generation (RAG) works in four steps:

1. **Chunk** each document into fixed-size pieces.
2. **Embed** each chunk into a vector.
3. **Retrieve** the chunks whose vectors are closest to the question's vector.
4. **Generate** the answer from those chunks.

No framework is used here: every step fits in a few lines, which makes the baseline fully transparent.

In [ ]:
from dotenv import load_dotenv

from src import config

load_dotenv(config.PROJECT_ROOT / ".env")
print(f"Domain: {config.DOMAIN} | run mode: {config.RUN_MODE.value}")

In [ ]:
from src.data import load_run_inputs
from src.usage_tracking import UsageLedger, get_ledger_path

SYSTEM_NAME = "naive_rag"
INDEX_NAME = config.INDEX_NAME_BY_SYSTEM[SYSTEM_NAME]

run_inputs = load_run_inputs(config.DOMAIN, config.RUN_MODE)
ledger = UsageLedger(get_ledger_path(run_inputs.run_directory))
index_directory = run_inputs.run_directory / "indexes" / INDEX_NAME
print(f"{len(run_inputs.questions)} questions, {len(run_inputs.documents)} documents")

## 2.1 Chunk

Chunks of 1,200 tokens with a 100-token overlap, the setting of the WildGraphBench paper. LightRAG and GraphRAG use the same values, so every system starts from identical pieces of text.

In [ ]:
import pandas as pd

from src.data import split_text_into_chunks

chunk_rows = [
    {"document_id": document.document_id, "chunk_index": chunk_index, "chunk_text": chunk_text}
    for document in run_inputs.documents.itertuples()
    for chunk_index, chunk_text in enumerate(
        split_text_into_chunks(
            document.text,
            chunk_size_tokens=config.CHUNK_SIZE_TOKENS,
            chunk_overlap_tokens=config.CHUNK_OVERLAP_TOKENS,
            encoding_name=config.TOKENIZER_ENCODING,
        )
    )
]
chunks = pd.DataFrame(chunk_rows)
print(f"{len(chunks)} chunks from {chunks['document_id'].nunique()} documents")
print(chunks["chunk_text"].iloc[0][:600])

## 2.2 Embed

Each chunk becomes a vector of 1,536 numbers. The index is built once and saved; deleting `index_directory` forces a rebuild. The cost of every call lands in the usage ledger under the `indexing` phase.

In [ ]:
import time

import numpy as np

from src.usage_tracking import Phase, create_embeddings, load_indexing_report, save_indexing_report, usage_scope

chunks_path = index_directory / "chunks.parquet"
embeddings_path = index_directory / "chunk_embeddings.npy"

if load_indexing_report(run_inputs.run_directory, INDEX_NAME) and embeddings_path.exists():
    print("Index found, reusing it.")
else:
    ledger.discard_records(INDEX_NAME, Phase.INDEXING)
    index_directory.mkdir(parents=True, exist_ok=True)
    start_time = time.perf_counter()
    with usage_scope(INDEX_NAME, Phase.INDEXING):
        chunk_embeddings = await create_embeddings(
            chunks["chunk_text"].tolist(), model=config.EMBEDDING_MODEL, ledger=ledger
        )
    indexing_time_seconds = time.perf_counter() - start_time
    chunks.to_parquet(chunks_path, index=False)
    np.save(embeddings_path, chunk_embeddings)
    save_indexing_report(
        run_inputs.run_directory,
        INDEX_NAME,
        indexing_time_seconds=indexing_time_seconds,
        document_count=len(run_inputs.documents),
        corpus_token_count=int(run_inputs.documents["token_count"].sum()),
    )

chunks = pd.read_parquet(chunks_path)
chunk_embeddings = np.load(embeddings_path)
chunk_embeddings = chunk_embeddings / np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
print(f"Embedding matrix: {chunk_embeddings.shape}")

## 2.3 Retrieve

Cosine similarity measures how aligned two vectors are. With unit-length vectors, it is a plain dot product. The number of chunks kept depends on the question type, as in the paper: 5 for fact questions, 10 for summaries.

In [ ]:
async def retrieve_top_k_chunks(question: str, top_k: int) -> pd.DataFrame:
    """Return the `top_k` chunks closest to the question, with their similarity score."""
    question_embedding = (await create_embeddings([question], model=config.EMBEDDING_MODEL, ledger=ledger))[0]
    question_embedding = question_embedding / np.linalg.norm(question_embedding)
    similarity_scores = chunk_embeddings @ question_embedding
    best_chunk_positions = np.argsort(-similarity_scores)[:top_k]
    return chunks.iloc[best_chunk_positions].assign(similarity=similarity_scores[best_chunk_positions])

A multi-fact question is the interesting test: its evidence is spread over several pages. The table below shows whether the retrieved chunks come from the pages the question actually cites.

In [ ]:
example_question = run_inputs.questions[run_inputs.questions["question_type"] == "multi_fact"].iloc[0]
print("Q:", example_question["question"])
print("Cited documents:", example_question["cited_document_ids"])

with usage_scope(SYSTEM_NAME, Phase.QUERY, question_id="inspection"):
    retrieved_chunks = await retrieve_top_k_chunks(
        example_question["question"], top_k=config.TOP_K_CHUNKS_BY_QUESTION_TYPE["multi_fact"]
    )
retrieved_chunks.assign(
    is_cited=retrieved_chunks["document_id"].isin(example_question["cited_document_ids"]),
    preview=retrieved_chunks["chunk_text"].str[:120],
)[["document_id", "similarity", "is_cited", "preview"]]

## 2.4 Generate

The retrieved chunks are pasted into a prompt, and the answer model writes the response. The instructions mirror those of LightRAG and GraphRAG: rely on the context only, and answer in the shared response style.

In [ ]:
from src.usage_tracking import create_chat_completion

NAIVE_RAG_SYSTEM_PROMPT = """You are a helpful assistant answering questions from the provided context.
Use only the information in the context. If the context does not contain the answer, say so.
Response format: {response_type}.

---Context---
{context}"""


async def answer_with_naive_rag(question: str, question_type: str) -> str:
    """Answer one question from the chunks closest to it."""
    retrieved_chunks = await retrieve_top_k_chunks(question, top_k=config.TOP_K_CHUNKS_BY_QUESTION_TYPE[question_type])
    context = "\n\n---\n\n".join(retrieved_chunks["chunk_text"])
    system_prompt = NAIVE_RAG_SYSTEM_PROMPT.format(response_type=config.RESPONSE_TYPE, context=context)
    return await create_chat_completion(
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
        model=config.ANSWER_MODEL,
        ledger=ledger,
        reasoning_effort=config.GENERATION_REASONING_EFFORT,
    )

## 2.5 Answer every question

The runner saves each answer as soon as it arrives. Rerunning the cell only processes the questions still missing.

In [ ]:
from src.question_runner import answer_all_questions

naive_rag_predictions = await answer_all_questions(
    answer_function=answer_with_naive_rag,
    questions=run_inputs.questions,
    system_name=SYSTEM_NAME,
    run_directory=run_inputs.run_directory,
    max_concurrent_questions=config.MAX_CONCURRENT_QUESTIONS,
)
print(naive_rag_predictions["pred_answer"].iloc[0][:800])

## 2.6 What it cost

Indexing only calls the embedding model, which is why vector RAG is cheap to build. Each question costs one embedding and one answer call.

In [ ]:
ledger_summary = ledger.summarize_by_system_and_phase()
ledger_summary[ledger_summary["system_name"].isin([INDEX_NAME, SYSTEM_NAME])]

## Next

Vector search sees each chunk in isolation. Notebook 03 adds structure: LightRAG turns the chunks into a graph of entities and relations.